In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import numpy as np
import matplotlib.pyplot as plt

from snudd.geometry import SolarAngles
from snudd.nsi import earth_matter, nsi_probabilities
from snudd.models import GeneralNSI, SM

from snudd.targets import nucleus_xe

In [3]:
nsi_model = GeneralNSI([[1, 0, 0],
                        [0, 1, 0],
                        [0, 0, 0]],
                        np.pi/4, 0)

nucleus_xe.update_model(nsi_model)


In [25]:
%%timeit
nucleus_xe.prepare_density()

114 ms ± 1.55 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [62]:
E_nus = np.geomspace(4e-3,1e1, 500) / 1e3

In [63]:
%%timeit
density_true = nucleus_xe._spec.density_calc.density(E_nus, '8B')


12.7 ms ± 208 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [61]:
%%timeit
density_interp = nucleus_xe._spec.nu_density_elements['8B'](E_nus)

133 μs ± 3.5 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [59]:
density_interp.shape

(12, 400)

## Testing earth evolution

In [5]:
t0 = 91
T  = 182*2

GranSasso = SolarAngles(latitude=42.47, t0=t0, T=T)
angles, weights = GranSasso.zenith_hist(bins=25)

In [29]:
DensityCalc = nsi_probabilities.DensityMatrixEarthCalculator(nsi_model)
# DensityCalc = nsi_probabilities.DensityMatrixEarthCalculator(nsi_model, cetas=[np.cos(np.pi/4)])

interpolated_rhos = DensityCalc.interpolate_earth_density_elements(cetas=np.cos(angles))


# # 
# import cProfile


# cProfile.run("DensityCalc.interpolate_earth_density_elements()")

In [22]:
print(angles/np.pi)

E_nus   = np.geomspace(3.4640e-3, 1.8784e1, 200) / 1e3
rho_els = interpolated_rhos['8B'][0](E_nus)

rhos = DensityCalc.matrix_from_elements(rho_els)
print(rhos.shape)

[0.10572381 0.12149488 0.13726594 0.15303701 0.16880808 0.18457915
 0.20035022 0.21612128 0.23189235 0.24766342 0.26343449 0.27920555
 0.29497662 0.31074769 0.32651876 0.34228983 0.35806089 0.37383196
 0.38960303 0.4053741  0.42114516 0.43691623 0.4526873  0.46845837
 0.48422944 0.5000005  0.51577157 0.53154264 0.54731371 0.56308477
 0.57885584 0.59462691 0.61039798 0.62616905 0.64194011 0.65771118
 0.67348225 0.68925332 0.70502438 0.72079545 0.73656652 0.75233759
 0.76810866 0.78387972 0.79965079 0.81542186 0.83119293 0.84696399
 0.86273506 0.87850613 0.8942772 ]
(200, 3, 3)


In [5]:
nucleus_xe.prepare_density()


E_nus = np.geomspace(4e-3,1e1, 500) / 1e3

nucleus_xe._spec.nu_density_elements["8B"][0](E_nus)

array([[ 5.49186869e-01,  5.49183116e-01,  5.49179303e-01, ...,
         3.12125890e-01,  3.11767133e-01,  3.11417217e-01],
       [-3.73480556e-18, -3.00057850e-18, -3.19978627e-19, ...,
         1.93141767e-17,  7.85368861e-18, -4.52335376e-18],
       [ 5.12518392e-02,  5.12466100e-02,  5.12412981e-02, ...,
        -2.79026554e-01, -2.79526382e-01, -2.80013893e-01],
       ...,
       [ 1.36550630e-02,  1.36543371e-02,  1.36535997e-02, ...,
        -3.21924227e-02, -3.22618062e-02, -3.23294798e-02],
       [ 2.35874816e-01,  2.35875417e-01,  2.35876027e-01, ...,
         2.73814987e-01,  2.73872404e-01,  2.73928406e-01],
       [ 5.35906823e-18,  2.41695354e-18, -5.94636781e-18, ...,
         2.12214226e-19,  7.88475005e-19,  3.53512131e-19]],
      shape=(12, 500))